In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Read Inside Airbnb Data").getOrCreate()

In [3]:
listings = spark.read.csv(
    "../data/listings.csv.gz",
    sep=",",
    quote='"',
    escape='"',
    header=True,
    inferSchema=True,
    multiLine=True,
    mode="PERMISSIVE"
)

In [4]:
listings.printSchema()

root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nullable = true)
 |-- host_listings_count: int

In [6]:
# 1. Get a non-null picture URL for any property ("picture_url" field)
# Select any non-null picture URL

listings.filter(listings.picture_url.isNotNull())\
    .select('picture_url').limit(1).show()

+--------------------+
|         picture_url|
+--------------------+
|https://a0.muscac...|
+--------------------+



In [7]:
# 2. Get number of properties that get more than 10 reviews per month

listings.filter(listings.reviews_per_month > 10).count()

66

In [ ]:
# 3. Get a property that has more bathrooms than bedrooms

listings.filter(listings.bathrooms > listings.bedrooms).select('name', 'bathrooms', 'bedrooms')\
    .show(20, truncate=False)

In [9]:
from pyspark.sql.functions import regexp_replace

# 4. Get 10 properties where the price is greater than 5,000. Collect the result as a Python list
# Remember to convert a price into a number first!
listings_with_price = listings\
    .withColumn('price_numeric', regexp_replace('price', '[$,]', '')\
    .cast('float'))

res = listings_with_price.filter(listings_with_price.price_numeric > 5000)\
    .select('name', 'price_numeric').collect()

res



[Row(name='Room in a cosy flat. Central, clean', price_numeric=8000.0),
 Row(name='Spacious Private Ground Floor Room', price_numeric=6309.0),
 Row(name='No Longer Available', price_numeric=53588.0),
 Row(name='Bright & airy DoubleBed with EnSuite in Zone 2!', price_numeric=74100.0),
 Row(name='The Apartments by The Sloane Club, Two Bedroom Apt', price_numeric=7377.0),
 Row(name='The Apartments by The Sloane Club, L 2 Bedroom Apt', price_numeric=7377.0),
 Row(name='Single room. 7ft x 9ft - Over looking garden', price_numeric=6523.0),
 Row(name='Close To London Eye (TUR)', price_numeric=6666.0),
 Row(name='Beautiful 2 BR flat in Kilburn with free parking', price_numeric=6000.0),
 Row(name='Semi-detached mews house in Knightsbridge.', price_numeric=7019.0),
 Row(name='Affordable Spacious  Room on the edge of the city', price_numeric=6000.0),
 Row(name='Henry’s Townhouse, London', price_numeric=6500.0),
 Row(name='City Suite', price_numeric=5353.0),
 Row(name='Hyde Park Suite', price_nume

In [10]:
# 5. Get a list of properties with the following characteristics:
# * price < 150
# * more than 20 reviews
# * review_scores_rating > 4.5
# Consider using the "&" operator

price_num_df = listings.withColumn('price_num', regexp_replace('price', '[$,]', '')\
                                   .cast('float'))

price_num_df.filter(
    (price_num_df.price_num < 1500)
    & (price_num_df.number_of_reviews > 20)
    & (price_num_df.review_scores_rating > 4.5))\
    .select('name', 'price_num', 'number_of_reviews', 'review_scores_rating')\
    .show(200, truncate=False)

+--------------------------------------------------+---------+-----------------+--------------------+
|name                                              |price_num|number_of_reviews|review_scores_rating|
+--------------------------------------------------+---------+-----------------+--------------------+
|Holiday London DB Room Let-on going               |70.0     |55               |4.85                |
|Bright Chelsea  Apartment. Chelsea!               |149.0    |97               |4.8                 |
|Very Central Modern 3-Bed/2 Bath By Oxford St W1  |411.0    |56               |4.77                |
|Kew Gardens 3BR house in cul-de-sac               |280.0    |116              |4.8                 |
|You are GUARANTEED to love this                   |90.0     |730              |4.87                |
|SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST        |61.0     |387              |4.77                |
|Short Term Home                                   |340.0    |42               |4.

In [11]:
# 6. Get a list of properties with the following characteristics:
# * price < 150 OR more than one bathroom
# Use the "|" operator to implement the OR operator

price_num_df.filter((price_num_df.price_num < 150) | (price_num_df.bathrooms > 1))\
    .select('name', 'price_num', 'bathrooms').show(50, truncate=False)

+--------------------------------------------------+---------+---------+
|name                                              |price_num|bathrooms|
+--------------------------------------------------+---------+---------+
|Holiday London DB Room Let-on going               |70.0     |1.0      |
|Bright Chelsea  Apartment. Chelsea!               |149.0    |1.0      |
|Very Central Modern 3-Bed/2 Bath By Oxford St W1  |411.0    |2.0      |
|Kew Gardens 3BR house in cul-de-sac               |280.0    |1.5      |
|You are GUARANTEED to love this                   |90.0     |0.0      |
|SUNNY ROOM PRIVATE BATHROOM PLUS BREAKFAST        |61.0     |1.0      |
|Short Term Home                                   |340.0    |2.0      |
|SPACIOUS ROOM IN CONTEMPORARY STYLE FLAT          |49.0     |1.0      |
|Room with a view, shared flat,  central  Bankside |96.0     |1.0      |
|You Will Save Money Here                          |71.0     |1.0      |
|Quiet Comfortable Room in Fulham                  

In [12]:
# 7. Get the highest listing price in this dataset
# Consider using the "max" function from "pyspark.sql.functions"

import pyspark.sql.functions as sf

price_num_df.select(sf.max(price_num_df.price_num)).show(20)

+--------------+
|max(price_num)|
+--------------+
|     1085147.0|
+--------------+



In [29]:
# 8. Get the name and a price of property with the highest number of reviews per month
# Try to use "collect" method to get the price first, and then use it in a "filter" call

res = price_num_df.select(sf.max(price_num_df.price_num).alias('max_price')).collect()
max_price = res[0]['max_price']
price_num_df.filter(price_num_df.price_num == max_price).select('name', 'price').show(truncate=False)

+---------------------------------------------+-------------+
|name                                         |price        |
+---------------------------------------------+-------------+
|Lux 2 Bed in Canary Wharf close to Excel & O2|$1,085,147.00|
+---------------------------------------------+-------------+



In [30]:
# 9. Get the number of hosts in the dataset
listings.select('host_name').distinct().count()

16673

In [31]:
# 10. Get listings with a first review in 2024
# Consider using the "year" function from "pyspark.sql.functions"

from pyspark.sql.functions import year

listings.filter(year(listings.first_review) == 2024)\
    .select('name', 'first_review').show(10, truncate=False)

+--------------------------------------------------+------------+
|name                                              |first_review|
+--------------------------------------------------+------------+
|Close to Wimbledon All England Tennis -huge double|2024-08-11  |
|one Double bed room with en-suite facilities      |2024-03-21  |
|Bridgerton inspired cottage core apartment        |2024-09-14  |
|Sm double room  with own bathroom                 |2024-06-04  |
|Central, modern pied-a-terre                      |2024-11-29  |
|Superlux flat in Knightsbridge                    |2024-01-01  |
|The Pink House, Notting Hill                      |2024-07-14  |
|Stylish garden flat in Hackney                    |2024-09-15  |
|Luxurious Flat in South Kensington                |2024-06-19  |
|Double Standard Room (Ensuite)                    |2024-09-01  |
+--------------------------------------------------+------------+
only showing top 10 rows
